# 06 — Temporal Agenda Dynamics

Agenda-setting theory (McCombs & Shaw 1972) is fundamentally about *temporal precedence*.
This notebook adds a temporal dimension to H1, answering:

1. **Is agenda distance stable over time?** Rolling JSD shows whether outlets converge or diverge during the study period.
2. **Which topics drive temporal variation?** Weekly time series of key divergent topics.
3. **Who leads whom?** Time-lagged correlations test whether alt outlets lead or follow Tagesschau on specific topics.

**Literature**:
- Vargo & Guo (2017, JMCQ): Granger causality on daily topic proportions — partisan media led mainstream agenda
- Field et al. (2018, EMNLP): Russian media agenda shifts over time (distraction hypothesis)

In [ ]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent.parent
OUTPUT_DIR = NOTEBOOK_DIR / "outputs"

sys.path.insert(0, str(NOTEBOOK_DIR))
sys.path.insert(0, str(PROJECT_ROOT / "1a_BERTopic"))

from modeling import load_iteration
from metrics import weekly_topic_proportions, time_lagged_correlations, rolling_jsd
from visualization import plot_topic_time_series, plot_rolling_jsd, plot_lag_correlation
from merged_outlets_analysis import THESIS_COLORS

v1 = load_iteration(OUTPUT_DIR, "v1")
print(f"Loaded: {v1.n_topics} topics, {len(v1.merged_articles):,} articles")

# Build display label lookup
label_map = dict(
    v1.merged_topic_info[["Topic", "DisplayLabel"]].dropna().values
)

In [ ]:
# Compute weekly topic proportions
weekly = weekly_topic_proportions(v1.merged_articles)
print(f"Weekly data: {len(weekly):,} rows")
print(f"Weeks covered: {weekly['week'].nunique()}")
print(f"Date range: {weekly['week'].min()} to {weekly['week'].max()}")
weekly.to_csv(OUTPUT_DIR / "v1" / "weekly_topic_proportions.csv", index=False)

## 1. Rolling JSD — Is agenda distance stable over time?

In [ ]:
fig_dir = OUTPUT_DIR / "v1" / "figures"

# Rolling JSD (4-week window)
roll = rolling_jsd(weekly, reference_label="Tagesschau", window=4)
roll.to_csv(OUTPUT_DIR / "v1" / "rolling_jsd.csv", index=False)

fig = plot_rolling_jsd(
    roll, colors=THESIS_COLORS,
    save_path=fig_dir / "h1_11_rolling_jsd.pdf",
)

## 2. Key topic time series

Track the most divergent topics over time: AfD/parties, Russia/Ukraine, Merz/chancellor, migration.

In [ ]:
# Key divergent topics identified from chi-squared analysis
key_topics = [
    (2, "AfD/BSW/Parties"),
    (4, "Russia/Ukraine (Russian perspective)"),
    (8, "Merz/Chancellor"),
    (10, "Asylum/Migration"),
    (1, "Ukraine/Putin/Selenskyj"),
    (6, "Social media/Youth"),
]

for topic_id, topic_name in key_topics:
    lbl = label_map.get(topic_id, topic_name)
    fig = plot_topic_time_series(
        weekly, topic_id, topic_label=lbl,
        reference_label="Tagesschau", colors=THESIS_COLORS,
        save_path=fig_dir / f"h1_12_timeseries_topic_{topic_id}.pdf",
    )

## 3. Time-lagged correlations — Who leads whom?

For each key topic, compute cross-correlations at lags -4 to +4 weeks.
- **Positive lag** = alt outlet LEADS Tagesschau (alt publishes first)
- **Negative lag** = alt outlet FOLLOWS Tagesschau
- **Peak at lag 0** = simultaneous coverage (reactive to same events)

In [ ]:
import pandas as pd

all_lag_results = []

for topic_id, topic_name in key_topics:
    lbl = label_map.get(topic_id, topic_name)
    lag_df = time_lagged_correlations(weekly, topic_id, max_lag=4)
    
    if len(lag_df) > 0:
        all_lag_results.append(lag_df)
        fig = plot_lag_correlation(
            lag_df, topic_label=lbl, colors=THESIS_COLORS,
            save_path=fig_dir / f"h1_13_lagcorr_topic_{topic_id}.pdf",
        )
        
        # Print summary: which lag has peak correlation per outlet?
        print(f"\n--- {lbl} (Topic {topic_id}) ---")
        for outlet in sorted(lag_df["outlet_label"].unique()):
            odata = lag_df[lag_df["outlet_label"] == outlet]
            peak = odata.loc[odata["correlation"].abs().idxmax()]
            sig = "*" if peak["p_value"] < 0.05 else ""
            direction = "LEADS" if peak["lag"] > 0 else ("FOLLOWS" if peak["lag"] < 0 else "SIMULTANEOUS")
            print(f"  {outlet:25s}  peak at lag={peak['lag']:+d}  r={peak['correlation']:+.3f}{sig}  ({direction})")

# Save all lag results
if all_lag_results:
    combined_lags = pd.concat(all_lag_results, ignore_index=True)
    combined_lags.to_csv(OUTPUT_DIR / "v1" / "time_lagged_correlations.csv", index=False)

## Summary

| Analysis | Key question | Output |
|----------|-------------|--------|
| Rolling JSD | Is distortion stable over time? | `h1_11_rolling_jsd.pdf` |
| Topic time series | How do key topics evolve? | `h1_12_timeseries_topic_*.pdf` |
| Lagged correlations | Who leads whom? | `h1_13_lagcorr_topic_*.pdf` |

### Interpretation guide
- **Stable rolling JSD**: Distortion is structural, not event-driven
- **Spiky rolling JSD**: Distortion fluctuates with news cycles
- **Peak at lag 0**: Outlets react to same events (not agenda-setting)
- **Peak at positive lag**: Alt outlet covers topic BEFORE Tagesschau picks it up
- **Peak at negative lag**: Alt outlet amplifies after Tagesschau reports